### Python Agent for Prisoners Delima

In [ ]:
import os
import json
# Example using a generic requests-based approach (adapt to Groq/Gemini SDK)
import requests 

In [ ]:
# 1. MEMORY: Keep track of the game state
class TrustAgent:
    def __init__(self, agent_name="MyAgent"):
        self.agent_name = agent_name
        # Stores the history: [{"round": 1, "my_move": "C", "opponent_move": "C", "opponent_msg": "Hello"}]
        self.history = []  

    # 2. REASONING: Construct the prompt
    def build_prompt(self, round_num, opponent_name, history):
        history_str = "\n".join([
            f"Round {h['round']}: You {h['my_move']}, Opponent {h['opponent_move']}. They said: '{h.get('opponent_msg', 'Nothing')}'"
            for h in history
        ])
        
        return f"""
        You are playing a 7-round Iterated Prisoner's Dilemma. 
        PAYOFFS: Mutual Cooperate (+3/+3), You Defect/They Cooperate (+5/0), Mutual Defect (+1/+1), You Cooperate/They Defect (0/+6).
        Current Round: {round_num} of 7.
        Opponent: {opponent_name}
        History so far:
        {history_str if history else "No previous rounds."}
        
        TASK: 
        1. Analyze the opponent's behavior and message.
        2. Decide your move: "COOPERATE" or "DEFECT".
        3. Write a short message (max 150 chars) to send to the opponent.
        
        Respond ONLY in this exact JSON format:
        {{
            "reasoning": "Brief explanation of your strategy this round",
            "message": "Your short message to the opponent upto 150 chars max",
            "decision": "COOPERATE" or "DEFECT"
        }}
        """

    # 3. ACTION: Call the LLM and parse the response
    def get_decision(self, round_num, opponent_name, opponent_msg=""):
        # Add current round's incoming message to history context if needed
        
        prompt = self.build_prompt(round_num, opponent_name, self.history)
        
        # --- REPLACE WITH YOUR CHOSEN FREE API (e.g., Groq, Gemini) ---
        # Example pseudo-code for API call:
        # response = requests.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}"}, json={"messages": [{"role": "user", "content": prompt}]})
        # llm_output = response.json()["choices"][0]["message"]["content"]
        
        # FOR TESTING: Mock response (Replace with actual API call)
        llm_output = '{"reasoning": "Testing", "message": "Let us cooperate.", "decision": "COOPERATE"}'
        
        try:
            # Parse the JSON response robustly
            result = json.loads(llm_output)
            decision = result["decision"].strip().upper()
            message = result["message"][:150] # Enforce 150 char limit
            
            # Ensure decision is valid
            if decision not in ["COOPERATE", "DEFECT"]:
                decision = "DEFECT" # Fallback
                
            return decision, message
            
        except Exception as e:
            # CRITICAL: Fallback so I don't exceed the 25s time limit or API crash
            print(f"Error parsing LLM response: {e}") # e is an object;
            return "DEFECT", "Error in reasoning." # This makes no sence how is the code valid

    # 4. UPDATE MEMORY: Called after the round resolves, record last round;
    def update_history(self, round_num, my_move, opponent_move, opponent_msg):
        self.history.append({
            "round": round_num,
            "my_move": my_move,
            "opponent_move": opponent_move,
            "opponent_msg": opponent_msg
        })

"""
Repuations carry along matchups so i have to remember what the apponent did in throuht out the tornament
"""